# Day 020 — Exercise 5: rag_answer

**Goal:** Implement `rag_answer(question, collection, model, n_results)` — the full RAG pipeline: retrieve → build_cited_prompt → ollama.chat. Returns `{'answer': str, 'sources': list[str]}`. **One embed + one LLM call** in the checks.

In [ ]:
import ollama
import chromadb

## Provided: all pipeline functions

In [ ]:
def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50) -> list[str]:
    words = text.split()
    step = chunk_size - overlap
    if step <= 0:
        step = 1
    chunks = []
    for i in range(0, len(words), step):
        chunk = " ".join(words[i : i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks


def embed_text(text: str, model: str = "nomic-embed-text") -> list[float]:
    return ollama.embeddings(model=model, prompt=text)["embedding"]


def build_index(
    docs: dict,
    collection_name: str = "second_brain",
    chunk_size: int = 300,
    overlap: int = 50,
):
    client = chromadb.Client()
    try:
        client.delete_collection(collection_name)
    except Exception:
        pass
    collection = client.create_collection(collection_name)
    for source, text in docs.items():
        chunks = chunk_text(text, chunk_size, overlap)
        ids, embeddings, documents, metadatas = [], [], [], []
        for i, chunk in enumerate(chunks):
            ids.append(f"{source}__{i}")
            embeddings.append(embed_text(chunk))
            documents.append(chunk)
            metadatas.append({"source": source, "chunk_index": i})
        if ids:
            collection.add(
                ids=ids, embeddings=embeddings,
                documents=documents, metadatas=metadatas,
            )
    return collection


def retrieve(query: str, collection, n_results: int = 3) -> list[dict]:
    if collection.count() == 0:
        return []
    emb = embed_text(query)
    actual_n = min(n_results, collection.count())
    results = collection.query(query_embeddings=[emb], n_results=actual_n)
    return [
        {"text": doc, "source": meta["source"], "distance": dist}
        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        )
    ]


def build_cited_prompt(question: str, chunks: list[dict]) -> str:
    context = "\n\n".join(
        f"[{i+1}] Source: {c['source']}\n{c['text']}"
        for i, c in enumerate(chunks)
    )
    return f"Context:\n{context}\n\nQuestion: {question}"


RAG_SYSTEM_PROMPT = (
    "You are a helpful assistant. Answer questions using ONLY the "
    "provided context. Cite the source numbers you used (e.g. [1], [2]). "
    "If the answer is not in the context, say 'I don't know.'"
)

## Your Implementation

In [ ]:
def rag_answer(
    question: str,
    collection,
    model: str = 'llama3.2',
    n_results: int = 3,
) -> dict:
    """
    Full RAG pipeline. Returns {'answer': str, 'sources': list[str]}.
    If no chunks found, returns {'answer': "I don't know.", 'sources': []}.
    """
    # TODO: chunks = retrieve(question, collection, n_results)
    # TODO: if not chunks: return guard response
    # TODO: prompt = build_cited_prompt(question, chunks)
    # TODO: ollama.chat with RAG_SYSTEM_PROMPT + user prompt
    # TODO: sources = list(dict.fromkeys(c['source'] for c in chunks))
    # TODO: return {'answer': str, 'sources': list}
    pass

## Check Your Work

In [ ]:
TEST_DOCS_5 = {
    "ai_notes.txt": (
        "Neural networks learn from data by adjusting weights during training. "
        "Deep learning uses many stacked layers to find complex patterns."
    ),
}

def _run_checks():
    total = 5
    passed = 0
    col = build_index(TEST_DOCS_5, collection_name='test_rag_020', chunk_size=20, overlap=3)
    result = None

    # Check 1: defined
    try:
        assert 'rag_answer' in globals()
        passed += 1; print('✅ Check 1: rag_answer defined')
    except Exception as e:
        print(f'❌ Check 1: {e}')

    # Check 2: empty collection guard (no LLM call)
    try:
        _client = chromadb.Client()
        try: _client.delete_collection('emptyragcol')
        except: pass
        _empty = _client.create_collection('emptyragcol')
        guard = rag_answer('test question', _empty)
        assert guard['answer'] == "I don't know.", f"got: {guard['answer']!r}"
        assert guard['sources'] == [], f"got: {guard['sources']}"
        passed += 1; print("✅ Check 2: empty collection returns I don't know.")
    except Exception as e:
        print(f'❌ Check 2: empty guard — {e}')

    # Check 3: returns dict with answer + sources (1 embed + 1 LLM call)
    try:
        result = rag_answer('What do neural networks do?', col)
        assert isinstance(result, dict), f'expected dict, got {type(result)}'
        assert 'answer' in result and 'sources' in result, f'missing keys: {list(result)}'
        passed += 1; print('✅ Check 3: returns dict with answer and sources')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: answer is a non-empty string
    try:
        assert result is not None, 'result is None (Check 3 failed)'
        assert isinstance(result['answer'], str) and len(result['answer']) > 0
        passed += 1; print('✅ Check 4: answer is a non-empty string')
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: sources is list[str]
    try:
        assert isinstance(result['sources'], list), f"expected list, got {type(result['sources'])}"
        for s in result['sources']:
            assert isinstance(s, str), f'source is not a string: {s!r}'
        passed += 1; print(f"✅ Check 5: sources is list[str] — {result['sources']}")
    except Exception as e:
        print(f'❌ Check 5: {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')

_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def rag_answer(
    question: str,
    collection,
    model: str = "llama3.2",
    n_results: int = 3,
) -> dict:
    chunks = retrieve(question, collection, n_results)
    if not chunks:
        return {"answer": "I don't know.", "sources": []}
    prompt = build_cited_prompt(question, chunks)
    response = ollama.chat(model=model, messages=[
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ])
    answer  = response["message"]["content"]
    sources = list(dict.fromkeys(c["source"] for c in chunks))
    return {"answer": answer, "sources": sources}
```

</details>